# Guia Completo de Manipulação de Arquivos em Python 3

*Para quem está começando e para quem já tem conhecimento intermediário · Com exemplos aplicados ao Django*

---

## Como usar este notebook

Continuamos a história da **loja virtual**. Agora o sistema precisa **registrar pedidos, guardar produtos e gerar relatórios em arquivos**.

- Execute as células **na ordem**, de cima para baixo (atalho: `Shift + Enter`). Na seção 4 os exemplos são sequenciais: o arquivo `pedidos.txt` nasce, cresce e é lido ao longo do notebook.
- Os comentários após `#` explicam cada linha e mostram o resultado esperado.
- Os blocos **Para ir além** trazem detalhes para quem já tem alguma experiência.
- Os exemplos de **Django** aparecem como código formatado (não executável), pois precisam de um projeto Django.
- Todos os arquivos criados ficam em uma pasta própria (`oficina_arquivos`), preparada na próxima célula. No final há uma célula opcional para apagá-la.

### Conteúdo

1. O que é manipulação de arquivos?
2. Características
3. Quando utilizar
4. Principais métodos
5. Erros mais comuns

## Preparação do ambiente

Vamos criar uma pasta só para os nossos exercícios, para que os arquivos criados aqui não se misturem com os seus outros arquivos. Pode executar a célula abaixo mais de uma vez sem problemas.

In [1]:
import os
from pathlib import Path

# Cria a pasta 'oficina_arquivos' e entra nela (só se ainda não estivermos lá)
if Path.cwd().name != 'oficina_arquivos':
    Path('oficina_arquivos').mkdir(exist_ok=True)
    os.chdir('oficina_arquivos')

print('Pasta de trabalho:', Path.cwd().name)  # oficina_arquivos

Pasta de trabalho: oficina_arquivos


---

# 1. O que é manipulação de arquivos?

Pense em um caderno de papel: para anotar algo, você precisa **abrir** o caderno, **ler ou escrever** e, no fim, **fechá-lo**. Manipular arquivos em programação é exatamente isso: usar o código para **criar, ler, alterar, copiar, mover e apagar** arquivos (textos, planilhas, imagens, PDFs...) guardados no computador ou no servidor.

Por que isso importa? Variáveis e listas vivem na memória e **desaparecem quando o programa termina**. Já os arquivos ficam gravados no disco: **os dados continuam lá depois**.

Toda manipulação de arquivo segue três etapas:

| Etapa | O que acontece | Analogia do caderno |
|---|---|---|
| 1. Abrir | O programa pede ao sistema acesso ao arquivo | Abrir o caderno |
| 2. Usar | Ler ou escrever o conteúdo | Ler ou anotar |
| 3. Fechar | O arquivo é liberado para o sistema | Fechar o caderno |

Seu primeiro contato:

In [2]:
# 1) Escrevendo: cria o arquivo 'pedidos.txt' e grava uma linha nele
with open('pedidos.txt', 'w', encoding='utf-8') as arquivo:
    arquivo.write('Pedido 1: caneta\n')  # o \n é a quebra de linha (como apertar Enter)

# 2) Lendo: abre o mesmo arquivo e recupera o que foi gravado
with open('pedidos.txt', 'r', encoding='utf-8') as arquivo:
    conteudo = arquivo.read()

print(conteudo)  # Pedido 1: caneta

Pedido 1: caneta



No código acima, `'w'` significa *write* (escrever) e `'r'` significa *read* (ler). Os modos são detalhados na seção 2.

## Caminhos: onde está o arquivo?

Para abrir um arquivo, o Python precisa saber **onde** ele está. Isso é o *caminho*:

```python
'pedidos.txt'                 # relativo: procura na pasta onde o programa foi executado
'relatorios/pedidos.txt'      # relativo: dentro da subpasta 'relatorios'
'/home/ana/loja/pedidos.txt'  # absoluto (Linux/Mac): endereço completo, a partir da raiz
'C:/loja/pedidos.txt'         # absoluto (Windows): o Python aceita a barra normal (/) também
```

---

# 2. Características

## 2.1 Persistente

Os dados gravados em um arquivo **sobrevivem** ao fim do programa, ao fechamento do terminal e até ao desligamento do computador. É a grande diferença em relação às variáveis.

## 2.2 Precisa ser aberto e fechado

Enquanto está aberto, o arquivo é um recurso do sistema operacional. Esquecer de fechá-lo pode causar perda de dados (que ficaram só em uma memória temporária) e travar o arquivo para outros programas. A solução é a instrução `with`, que fecha o arquivo sozinha, **mesmo se acontecer um erro**:

In [3]:
with open('pedidos.txt', 'r', encoding='utf-8') as arquivo:
    conteudo = arquivo.read()

# Aqui fora do 'with', o arquivo já foi fechado automaticamente
print(arquivo.closed)  # True

True


## 2.3 O modo de abertura define o que você pode fazer

| Modo | Significado | Se o arquivo não existe | Se já existe |
|---|---|---|---|
| `'r'` | Ler (é o padrão) | Erro | Lê normalmente |
| `'w'` | Escrever | Cria | **Apaga todo o conteúdo** |
| `'a'` | Acrescentar (*append*) | Cria | Escreve no final |
| `'x'` | Criar exclusivamente | Cria | Erro |
| `'r+'` | Ler e escrever | Erro | Lê e escreve, sem apagar |

> **Atenção:** o modo `'w'` **apaga todo o conteúdo** de um arquivo que já existe, sem perguntar. Para acrescentar ao final, use `'a'`.

## 2.4 Texto ou binário

Arquivos de texto (`.txt`, `.csv`, `.json`) trabalham com `str`. Já imagens, PDFs e planilhas Excel são **binários** e trabalham com `bytes`. Para binário, acrescente `b` ao modo (`'rb'`, `'wb'`) e **não** informe `encoding`.

In [4]:
with open('dados.bin', 'wb') as arquivo:  # 'b' = binário
    arquivo.write(b'ABC')                 # em binário, gravamos bytes (note o prefixo b)

with open('dados.bin', 'rb') as arquivo:
    print(arquivo.read())                 # b'ABC'

b'ABC'


## 2.5 Depende da codificação (*encoding*)

Texto vira bytes no disco por meio de uma *codificação*. O padrão universal é o **UTF-8**, que representa bem acentos (á, ç, ã) e emojis. Informe sempre `encoding='utf-8'` em arquivos de texto: em alguns sistemas (como o Windows) o padrão pode ser outro. Veja o que acontece quando lemos com a codificação errada:

In [5]:
with open('acentos.txt', 'w', encoding='utf-8') as arquivo:
    arquivo.write('lápis')

with open('acentos.txt', 'r', encoding='utf-8') as arquivo:
    print(arquivo.read())  # lápis   -> codificação correta

with open('acentos.txt', 'r', encoding='latin-1') as arquivo:
    print(arquivo.read())  # lÃ¡pis  -> codificação errada: caracteres estranhos

lápis
lÃ¡pis


## 2.6 Leitura sequencial, com cursor

O arquivo aberto tem um **cursor** (a posição atual), que avança conforme você lê ou escreve, como um dedo acompanhando uma linha de texto. Por isso, ler duas vezes seguidas devolve texto vazio na segunda:

In [6]:
with open('pedidos.txt', 'r', encoding='utf-8') as arquivo:
    primeira = arquivo.read()  # lê tudo; o cursor vai parar no final
    segunda = arquivo.read()   # tenta ler de novo, mas não há mais nada à frente

print(primeira)        # Pedido 1: caneta
print(repr(segunda))   # '' -> texto vazio

Pedido 1: caneta

''


## 2.7 Pode falhar (e você deve se preparar)

Arquivos dependem do mundo externo: o arquivo pode não existir, a pasta pode estar protegida (`PermissionError`) ou o caminho pode apontar para uma pasta (`IsADirectoryError`). Use `try/except`:

In [7]:
# Tentando ler um arquivo que não existe
try:
    with open('nao_existe.txt', 'r', encoding='utf-8') as arquivo:
        print(arquivo.read())
except FileNotFoundError:
    print('Arquivo não encontrado. Verifique o nome e a pasta.')  # esta linha será executada

# O modo 'x' só cria arquivos NOVOS: protege contra sobrescrever sem querer
try:
    with open('pedidos.txt', 'x', encoding='utf-8') as arquivo:  # o pedidos.txt da seção 1 já existe
        arquivo.write('Isto não será gravado')
except FileExistsError:
    print('O arquivo já existe; nada foi alterado.')  # esta linha será executada

Arquivo não encontrado. Verifique o nome e a pasta.
O arquivo já existe; nada foi alterado.


## 2.8 Os caminhos variam conforme o sistema

Windows, Linux e Mac escrevem caminhos de formas diferentes. O módulo `pathlib` cuida disso para você:

In [8]:
from pathlib import Path

caminho = Path('relatorios') / 'pedidos.txt'  # o operador / junta as partes do caminho
print(caminho)  # relatorios/pedidos.txt (no Windows, aparece com barra invertida)

relatorios/pedidos.txt


---

# 3. Quando utilizar

Use arquivos quando você precisa:

- **guardar dados simples** que devem sobreviver ao programa (configurações, listas, anotações);
- **registrar logs** de eventos e erros;
- **importar ou exportar dados** em formatos como CSV e JSON;
- **receber arquivos de usuários** (fotos, comprovantes, PDFs) e **entregar downloads** (relatórios, notas fiscais);
- fazer **backups** e organizar pastas.

E quando preferir um banco de dados?

| Situação | Melhor escolha |
|---|---|
| Configurações simples, logs, exportar/importar CSV e JSON | **Arquivo** |
| Fotos, comprovantes e PDFs enviados por usuários | **Arquivo** (no Django: `FileField`) |
| Muitos usuários lendo e escrevendo ao mesmo tempo | **Banco de dados** |
| Buscas, filtros e relações entre dados (clientes, pedidos, produtos) | **Banco de dados** (no Django: ORM) |
| Pagamentos e estoque, que exigem consistência e transações | **Banco de dados** |

## A regra de ouro no Django

No Django, o **arquivo fica no disco** (ou em um serviço de armazenamento na nuvem) e o **banco de dados guarda apenas o caminho** dele. É exatamente isso que o `FileField` faz:

```python
# models.py
from django.db import models

class Pedido(models.Model):
    cliente = models.CharField(max_length=100)
    # O arquivo vai para MEDIA_ROOT/comprovantes/; o banco guarda só o caminho
    comprovante = models.FileField(upload_to='comprovantes/')
```

---

# 4. Principais métodos

Vamos construir o registro de pedidos da loja, **na ordem**. Os primeiros métodos pertencem ao *objeto de arquivo* devolvido por `open()`:

| Método | O que faz |
|---|---|
| `open(caminho, modo)` | Abre um arquivo e devolve o objeto de arquivo |
| `close()` | Fecha o arquivo |
| `write(texto)` | Escreve um texto |
| `writelines(lista)` | Escreve vários textos de uma vez |
| `read(n)` | Lê tudo (ou apenas `n` caracteres) |
| `readline()` | Lê uma linha |
| `readlines()` | Lê todas as linhas e devolve uma **lista** |
| `tell()` | Informa a posição do cursor |
| `seek(posicao)` | Move o cursor para outra posição |

## Abrir e fechar

### 4.1 `open()`: abre o arquivo

In [9]:
# Sintaxe: open(caminho, modo, encoding='utf-8')
arquivo = open('pedidos.txt', 'w', encoding='utf-8')  # 'w' cria o arquivo (ou limpa, se já existir)

print(arquivo.name)    # pedidos.txt
print(arquivo.mode)    # w
print(arquivo.closed)  # False -> o arquivo está aberto

pedidos.txt
w
False


Agora o `pedidos.txt` existe, mas está **vazio**. Como abrimos sem o `with`, fechar é responsabilidade nossa.

### 4.2 `close()`: fecha o arquivo

In [10]:
# Sintaxe: arquivo.close()
arquivo.close()        # fecha o arquivo aberto no exemplo anterior
print(arquivo.closed)  # True

# arquivo.write('x')   # se descomentar, dá ValueError: I/O operation on closed file

True


### 4.3 `with`: abre e fecha automaticamente

Não é um método, mas é a **forma recomendada** de usar `open()`. Daqui em diante, todos os exemplos usam `with`.

In [11]:
with open('pedidos.txt', 'w', encoding='utf-8') as arquivo:
    print(arquivo.closed)  # False -> dentro do with, o arquivo está aberto

print(arquivo.closed)      # True  -> ao sair do with, foi fechado sozinho

False
True


## Escrever

### 4.4 `write()`: escreve um texto

In [12]:
# Sintaxe: arquivo.write(texto)
with open('pedidos.txt', 'w', encoding='utf-8') as arquivo:
    quantidade = arquivo.write('Pedido 1: caneta\n')  # devolve quantos caracteres foram gravados
    arquivo.write('Pedido 2: caderno\n')  # sem o \n, os pedidos ficariam todos na mesma linha

print(quantidade)  # 17

17


O arquivo agora contém duas linhas. O `write()` só aceita texto: para gravar números, converta antes, por exemplo com `str(10)` ou um f-string (`f'Total: {10}'`).

**Para ir além:** os dados gravados passam primeiro por uma memória temporária (*buffer*). O `close()` (e portanto o `with`) descarrega esse buffer no disco; `arquivo.flush()` força isso sem fechar o arquivo.

### 4.5 `writelines()`: escreve uma lista de textos

In [13]:
# Sintaxe: arquivo.writelines(lista_de_textos)
novos = ['Pedido 3: borracha\n', 'Pedido 4: lápis\n']

with open('pedidos.txt', 'a', encoding='utf-8') as arquivo:  # 'a' = acrescenta no FINAL, sem apagar
    arquivo.writelines(novos)  # grava cada item da lista (NÃO adiciona o \n sozinho)

print('Pedidos gravados!')

Pedidos gravados!


O `pedidos.txt` agora tem **4 linhas**: pedidos 1, 2, 3 e 4.

## Ler

### 4.6 `read()`: lê tudo de uma vez

In [14]:
# Sintaxe: arquivo.read(quantidade)  -> a quantidade é opcional
with open('pedidos.txt', 'r', encoding='utf-8') as arquivo:
    conteudo = arquivo.read()  # devolve TODO o conteúdo em um único texto

print(conteudo)
# Pedido 1: caneta
# Pedido 2: caderno
# Pedido 3: borracha
# Pedido 4: lápis

with open('pedidos.txt', 'r', encoding='utf-8') as arquivo:
    print(arquivo.read(8))  # Pedido 1 -> lê só os 8 primeiros caracteres

Pedido 1: caneta
Pedido 2: caderno
Pedido 3: borracha
Pedido 4: lápis

Pedido 1


Cuidado: `read()` carrega o arquivo inteiro na memória. Para arquivos muito grandes, prefira ler linha a linha (4.9).

### 4.7 `readline()`: lê uma linha por vez

In [15]:
# Sintaxe: arquivo.readline()
with open('pedidos.txt', 'r', encoding='utf-8') as arquivo:
    primeira = arquivo.readline()  # 'Pedido 1: caneta\n'
    segunda = arquivo.readline()   # 'Pedido 2: caderno\n' (o cursor avançou)

print(primeira.strip())  # Pedido 1: caneta -> strip() remove o \n do final
print(segunda.strip())   # Pedido 2: caderno

Pedido 1: caneta
Pedido 2: caderno


Quando o arquivo acaba, o `readline()` devolve um texto vazio (`''`).

### 4.8 `readlines()`: lê todas as linhas em uma lista

Lembra das listas? O `readlines()` devolve uma, com **uma linha por item**:

In [16]:
# Sintaxe: arquivo.readlines()
with open('pedidos.txt', 'r', encoding='utf-8') as arquivo:
    linhas = arquivo.readlines()

print(linhas)
# ['Pedido 1: caneta\n', 'Pedido 2: caderno\n', 'Pedido 3: borracha\n', 'Pedido 4: lápis\n']
print(len(linhas))  # 4 -> quatro pedidos registrados

['Pedido 1: caneta\n', 'Pedido 2: caderno\n', 'Pedido 3: borracha\n', 'Pedido 4: lápis\n']
4


### 4.9 Percorrendo o arquivo com `for`: a forma mais eficiente

O próprio arquivo é percorrível: o Python lê **uma linha por vez**, sem carregar tudo na memória. Ideal para arquivos grandes.

In [17]:
with open('pedidos.txt', 'r', encoding='utf-8') as arquivo:
    for numero, linha in enumerate(arquivo, start=1):  # enumerate numera as linhas a partir de 1
        print(f'{numero} - {linha.strip()}')
# 1 - Pedido 1: caneta
# 2 - Pedido 2: caderno
# 3 - Pedido 3: borracha
# 4 - Pedido 4: lápis

1 - Pedido 1: caneta
2 - Pedido 2: caderno
3 - Pedido 3: borracha
4 - Pedido 4: lápis


## Controlar o cursor

### 4.10 `tell()` e `seek()`: onde estou e para onde vou

In [18]:
# Sintaxe: arquivo.tell()  e  arquivo.seek(posicao)
with open('pedidos.txt', 'r', encoding='utf-8') as arquivo:
    print(arquivo.tell())  # 0 -> o cursor começa no início

    arquivo.readline()     # lê a primeira linha
    print(arquivo.tell())  # 17 -> cursor depois da 1ª linha (no Windows pode ser 18)

    arquivo.seek(0)        # volta o cursor para o início
    print(arquivo.readline().strip())  # Pedido 1: caneta -> leu de novo

0
17
Pedido 1: caneta


**Para ir além:** em arquivos de texto, use `seek()` apenas com `0` ou com um valor obtido antes pelo `tell()`. Posições arbitrárias só são seguras em arquivos binários.

## Gerenciar arquivos e pastas (`pathlib` e `shutil`)

Agora saímos de **dentro** do arquivo e passamos a gerenciá-lo: verificar, copiar, mover, listar e apagar. O módulo `pathlib` é a forma moderna de fazer isso.

### 4.11 `Path.exists()`: verifica se existe

In [19]:
from pathlib import Path

arquivo = Path('pedidos.txt')  # só representa o caminho; não abre nada

print(arquivo.exists())                 # True  -> o arquivo existe
print(arquivo.is_file())                # True  -> é um arquivo (e não uma pasta)
print(arquivo.name)                     # pedidos.txt
print(arquivo.suffix)                   # .txt  -> a extensão
print(Path('nao_existe.txt').exists())  # False

True
True
pedidos.txt
.txt
False


### 4.12 `read_text()` e `write_text()`: ler e escrever em uma linha

Atalhos que **abrem, leem/escrevem e fecham** de uma só vez. Perfeitos para arquivos pequenos.

In [20]:
# Lê o arquivo inteiro e conta quantos pedidos existem
total = arquivo.read_text(encoding='utf-8').count('Pedido')
print(total)  # 4

# Cria (ou sobrescreve) um resumo
resumo = Path('resumo.txt')
resumo.write_text(f'Total de pedidos: {total}\n', encoding='utf-8')
print(resumo.read_text(encoding='utf-8'))  # Total de pedidos: 4

4
Total de pedidos: 4



### 4.13 `mkdir()`: cria uma pasta

In [21]:
pasta = Path('relatorios')
pasta.mkdir(exist_ok=True)  # exist_ok=True evita erro se a pasta já existir
print(pasta.exists())       # True

# parents=True cria também as pastas intermediárias:
# Path('relatorios/2026/outubro').mkdir(parents=True, exist_ok=True)

True


### 4.14 `shutil.copy()` e `shutil.move()`: copiar e mover

In [22]:
import shutil

# Copia: o original continua no lugar. Este é o nosso backup.
shutil.copy(arquivo, pasta / 'pedidos_backup.txt')

# Move: o arquivo sai do lugar antigo e vai para o novo
shutil.move(resumo, pasta / 'resumo.txt')

print(arquivo.exists())  # True  -> o pedidos.txt continua aqui (foi copiado)
print(resumo.exists())   # False -> o resumo.txt foi movido para a pasta

True
False


### 4.15 `rename()`: renomeia

In [23]:
(pasta / 'resumo.txt').rename(pasta / 'resumo_pedidos.txt')  # só troca o nome
print((pasta / 'resumo_pedidos.txt').exists())  # True

True


### 4.16 `iterdir()` e `glob()`: listam o conteúdo de uma pasta

In [24]:
# iterdir() percorre tudo o que existe dentro da pasta
for item in pasta.iterdir():
    print(item.name)
# pedidos_backup.txt   (a ordem pode variar)
# resumo_pedidos.txt

# glob() filtra por um padrão: '*.txt' = qualquer nome terminado em .txt
nomes = sorted(p.name for p in pasta.glob('*.txt'))  # sorted() deixa em ordem alfabética
print(nomes)  # ['pedidos_backup.txt', 'resumo_pedidos.txt']

resumo_pedidos.txt
pedidos_backup.txt
['pedidos_backup.txt', 'resumo_pedidos.txt']


### 4.17 `unlink()` e `rmdir()`: apagam arquivos e pastas

In [25]:
(pasta / 'resumo_pedidos.txt').unlink()                 # apaga um arquivo
(pasta / 'pedidos_backup.txt').unlink(missing_ok=True)  # missing_ok=True: sem erro se não existir
pasta.rmdir()                                           # remove a pasta (só se estiver vazia)

print(pasta.exists())  # False

False


> **Cuidado:** apagar com `unlink()` **não envia para a lixeira**: é definitivo. Para apagar uma pasta com tudo dentro existe `shutil.rmtree()`, que deve ser usado com muita atenção.

**Para ir além:** você verá código antigo usando `os.path.exists()`, `os.remove()` e `os.listdir()`. Funcionam, mas em código novo o `pathlib` é mais legível e portátil.

## Formatos comuns: CSV e JSON

### 4.18 `csv`: planilhas simples

O CSV guarda uma tabela em texto, com valores separados por vírgula. Vamos guardar o catálogo da loja:

In [26]:
import csv

produtos = [
    ['produto', 'preco'],  # a primeira linha é o cabeçalho
    ['caneta', 5.0],
    ['caderno', 12.5],
]

# newline='' evita linhas em branco extras no Windows
with open('produtos.csv', 'w', newline='', encoding='utf-8') as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerows(produtos)  # grava todas as linhas de uma vez

# Lendo: o DictReader transforma cada linha em um dicionário
with open('produtos.csv', 'r', newline='', encoding='utf-8') as arquivo:
    for linha in csv.DictReader(arquivo):
        print(linha['produto'], linha['preco'])
# caneta 5.0
# caderno 12.5

caneta 5.0
caderno 12.5


Atenção: tudo o que é lido de um CSV vem como **texto**. O preço `'5.0'` precisa de `float()` antes de entrar em contas.

### 4.19 `json`: dados estruturados

O JSON guarda dicionários e listas, e é o formato padrão de configurações e APIs:

In [27]:
import json

carrinho = {'cliente': 'Ana', 'itens': ['caneta', 'caderno'], 'total': 17.5}

with open('carrinho.json', 'w', encoding='utf-8') as arquivo:
    # ensure_ascii=False mantém os acentos; indent=2 deixa o arquivo legível
    json.dump(carrinho, arquivo, ensure_ascii=False, indent=2)

with open('carrinho.json', 'r', encoding='utf-8') as arquivo:
    dados = json.load(arquivo)  # lê o arquivo e devolve um dicionário Python

print(dados['itens'])  # ['caneta', 'caderno']
print(dados['total'])  # 17.5

['caneta', 'caderno']
17.5


## Arquivos no Django

> **Nota:** as células desta parte são código formatado (não executável), pois precisam de um projeto Django. Copie os trechos para o seu projeto.

As APIs abaixo são estáveis e funcionam do mesmo jeito no Django 6. Para o `Pedido` e o `FileField` da seção 3 funcionarem, configure onde os arquivos enviados serão guardados:

```python
# settings.py
MEDIA_URL = '/media/'             # endereço público dos arquivos enviados
MEDIA_ROOT = BASE_DIR / 'media'   # pasta no disco (BASE_DIR é um Path, por isso o / funciona)
```

### 4.20 `request.FILES`: receber um arquivo enviado pelo usuário

```python
# views.py
from django.shortcuts import render, redirect
from .models import Pedido

def enviar_comprovante(request):
    if request.method == 'POST':
        Pedido.objects.create(
            cliente=request.POST['cliente'],           # campo de texto do formulário
            comprovante=request.FILES['comprovante'],  # o arquivo enviado pelo usuário
        )
        return redirect('carrinho')  # use o nome de uma rota do seu projeto
    return render(request, 'loja/enviar.html')
```

```html
<!-- loja/enviar.html -->
<form method='post' enctype='multipart/form-data'>  <!-- o enctype é obrigatório para enviar arquivos! -->
  {% csrf_token %}
  <input type='text' name='cliente'>
  <input type='file' name='comprovante'>
  <button type='submit'>Enviar</button>
</form>
```

Em projetos reais, use um `ModelForm` para validar os dados. E, só em **desenvolvimento**, adicione ao `urls.py` o que permite abrir os arquivos enviados no navegador:

```python
# urls.py (apenas para desenvolvimento)
from django.conf import settings
from django.conf.urls.static import static

urlpatterns += static(settings.MEDIA_URL, document_root=settings.MEDIA_ROOT)
```

### 4.21 Atributos e métodos do campo de arquivo

```python
pedido = Pedido.objects.get(pk=1)  # um pedido que já tem comprovante

print(pedido.comprovante.name)  # comprovantes/nota.pdf -> caminho relativo ao MEDIA_ROOT
print(pedido.comprovante.url)   # /media/comprovantes/nota.pdf -> use em <a> e <img> nos templates
print(pedido.comprovante.size)  # tamanho em bytes

with pedido.comprovante.open('rb') as arquivo:  # abre como qualquer outro arquivo
    primeiros_bytes = arquivo.read(4)

pedido.comprovante.delete(save=True)  # apaga o arquivo do disco E atualiza o registro no banco
```

No template, o link para o arquivo é `{{ pedido.comprovante.url }}`.

### 4.22 `default_storage`: salvar e consultar arquivos gerados pelo sistema

Para arquivos que **o seu código cria** (um relatório, por exemplo), use o `default_storage`. Ele esconde *onde* o arquivo fica (disco local ou nuvem), então o código não muda se você trocar o armazenamento:

```python
from django.core.files.base import ContentFile
from django.core.files.storage import default_storage

conteudo = 'Total de pedidos: 4\n'.encode('utf-8')  # texto -> bytes
nome = default_storage.save('relatorios/pedidos.txt', ContentFile(conteudo))
print(nome)  # relatorios/pedidos.txt (se já existisse, o Django criaria um nome diferente)

print(default_storage.exists(nome))  # True
print(default_storage.url(nome))     # /media/relatorios/pedidos.txt

# default_storage.delete(nome) apagaria o arquivo
```

### 4.23 `FileResponse`: entregar um arquivo para download

```python
from django.core.files.storage import default_storage
from django.http import FileResponse

def baixar_relatorio(request):
    arquivo = default_storage.open('relatorios/pedidos.txt', 'rb')  # abre em modo binário
    # as_attachment=True força o download; o Django fecha o arquivo ao terminar o envio
    return FileResponse(arquivo, as_attachment=True, filename='pedidos.txt')
```

**Para ir além (segurança):**

- Nunca monte caminhos de arquivo diretamente a partir de texto digitado pelo usuário: ele poderia usar `../` para acessar pastas que não deveria.
- Valide a **extensão e o tamanho** dos arquivos enviados, e não confie apenas no nome informado.
- O `static()` do `urls.py` serve para desenvolvimento. Em produção, entregue os arquivos de `MEDIA_ROOT` pelo servidor web ou por um serviço de armazenamento.

---

# 5. Erros mais comuns

| Erro | Por que acontece | Como evitar |
|---|---|---|
| `FileNotFoundError` | Nome errado ou o programa foi executado em outra pasta | Confira o caminho com `Path(...).exists()`; no Django, parta do `BASE_DIR` |
| Caracteres estranhos (`lÃ¡pis`) ou `UnicodeDecodeError` | Faltou informar a codificação | Use sempre `encoding='utf-8'` |
| Dados antigos sumiram | O modo `'w'` apaga o conteúdo existente | Use `'a'` para acrescentar ou `'x'` para só criar arquivos novos |
| `ValueError: I/O operation on closed file` | Usar o arquivo depois de sair do `with` | Faça todo o trabalho dentro do bloco `with` |
| Linhas em branco extras ao imprimir | A linha lida já termina com `\n` e o `print` adiciona outro | Use `linha.strip()` ou `print(linha, end='')` |
| `TypeError` ao usar `write()` | O `write()` só aceita texto | Converta antes: `str(numero)` ou f-string |
| `PermissionError` | Sem permissão na pasta, ou arquivo aberto em outro programa | Verifique as permissões e feche o arquivo no outro programa |
| Computador lento com arquivo enorme | O `read()` carrega tudo na memória | Percorra o arquivo com `for linha in arquivo` |

---

# Próximos passos

Depois de dominar a manipulação de arquivos, vale estudar **tratamento de exceções** em profundidade, o módulo **`logging`** (a forma profissional de registrar logs), bibliotecas como **`openpyxl`** (planilhas Excel) e **`Pillow`** (imagens, necessária para o `ImageField` do Django), além de **armazenamento na nuvem** com `django-storages`.

---

## Limpeza (opcional)

A célula abaixo apaga a pasta `oficina_arquivos` e tudo o que foi criado nos exercícios. Por segurança, ela **não faz nada** até que você troque `False` por `True`.

In [28]:
import os
import shutil
from pathlib import Path

CONFIRMAR_LIMPEZA = False  # mude para True e execute a célula para apagar a pasta de exercícios

if CONFIRMAR_LIMPEZA and Path.cwd().name == 'oficina_arquivos':
    os.chdir('..')                     # sai da pasta antes de apagá-la
    shutil.rmtree('oficina_arquivos')  # apaga a pasta e tudo o que há dentro
    print('Pasta de exercícios removida.')
else:
    print('Limpeza não executada. Mude CONFIRMAR_LIMPEZA para True se quiser apagar os arquivos.')

Limpeza não executada. Mude CONFIRMAR_LIMPEZA para True se quiser apagar os arquivos.
